Implement scaled dot-product attention from scratch using NumPy.
- Build a transformer encoder in PyTorch and train it on a small dataset.
- Modify an existing Transformer model to use adaptive attention span.
- Train a Transformer on time-series forecasting and compare it with an RNN.

In [2]:
import numpy as np

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    
    d_k = Q.shape[-1]
    
    # attention scores
    scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(d_k)
    
    # apply mask for decoders (give input of mask as non none)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    # attention weights
    attWeights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attWeights = attWeights / np.sum(attWeights, axis=-1, keepdims=True)
    
    # multiply by values
    output = np.matmul(attWeights, V)
    
    return output, attWeights

In [4]:
batch_size = 2
seq_len = 3
d_k = 4
d_v = 5

Q = np.random.randn(batch_size, seq_len, d_k)
K = np.random.randn(batch_size, seq_len, d_k)
V = np.random.randn(batch_size, seq_len, d_v)

In [8]:
output, attWeights = scaled_dot_product_attention(Q, K, V)
print("Output shape:", output.shape)
print("Output:", output)
print("Attention weights shape:", attWeights.shape)
print("Attention weights:", attWeights)

Output shape: (2, 3, 5)
Output: [[[-1.42148152e+00  3.84979767e-01  9.21362839e-01 -8.78684512e-02
    1.06161980e+00]
  [-1.64137456e+00  6.51462761e-01  8.32280627e-01  8.04927848e-01
    1.57245651e+00]
  [-1.46116096e+00  3.05850291e-01  9.15691342e-01  2.46312424e-01
    1.21558843e+00]]

 [[ 7.27062869e-04 -3.23833866e-01 -3.74135282e-01  8.03825453e-01
   -1.98929281e+00]
  [ 7.85334162e-03 -1.08886493e+00 -4.41919935e-01  1.17788241e+00
   -1.48274450e+00]
  [ 1.46243192e-01 -1.44202790e-01 -4.50525077e-01  8.18840224e-01
   -2.34937256e+00]]]
Attention weights shape: (2, 3, 3)
Attention weights: [[[0.35523072 0.37871757 0.2660517 ]
  [0.23091821 0.07814155 0.69094024]
  [0.25518154 0.32612868 0.41868978]]

 [[0.53381806 0.30176814 0.1644138 ]
  [0.29808566 0.19401223 0.50790211]
  [0.77125734 0.16207149 0.06667118]]]


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [22]:
def multihead_attention(q, k, v, nhead, dropout=0.1):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k))
    attn_weights = F.softmax(scores, dim=-1)
    attn_weights = F.dropout(attn_weights, p=dropout)
    output = torch.matmul(attn_weights, v)
    return output, attn_weights

In [23]:
def feed_forward(x, dim_feedforward, dropout=0.1):
    x = F.relu(nn.Linear(x.size(-1), dim_feedforward)(x))
    x = F.dropout(x, p=dropout)
    x = nn.Linear(dim_feedforward, x.size(-1))(x)
    return x

In [24]:
def layer_norm(x, normalized_shape):
    return nn.LayerNorm(normalized_shape)(x)

In [25]:
def encoder_layer(x, d_model, nhead, dim_feedforward, dropout=0.1):
    # Self-attention
    attn_output, _ = multihead_attention(x, x, x, nhead, dropout)
    x = x + F.dropout(attn_output, p=dropout)
    x = layer_norm(x, d_model)
    
    # Feedforward
    ff_output = feed_forward(x, dim_feedforward, dropout)
    x = x + F.dropout(ff_output, p=dropout)
    x = layer_norm(x, d_model)
    
    return x

In [26]:
def transformer_encoder(x, num_layers, d_model, nhead, dim_feedforward, dropout=0.1):
    for _ in range(num_layers):
        x = encoder_layer(x, d_model, nhead, dim_feedforward, dropout)
    return x

In [27]:
def create_dummy_data(num_samples=100, seq_len=10, d_model=64):
    data = torch.randn(num_samples, seq_len, d_model)
    targets = torch.randn(num_samples, d_model)
    return TensorDataset(data, targets)

In [28]:
def train_transformer_encoder():
    # Hyperparameters
    d_model = 64
    nhead = 4
    dim_feedforward = 128
    num_layers = 3
    batch_size = 16
    num_epochs = 10
    
    # Create model parameters
    # Note: Normally we'd use classes to manage these, but we'll create them manually
    attention_weights = [{
        'q_proj': nn.Parameter(torch.randn(d_model, d_model)),
        'k_proj': nn.Parameter(torch.randn(d_model, d_model)),
        'v_proj': nn.Parameter(torch.randn(d_model, d_model)),
        'out_proj': nn.Parameter(torch.randn(d_model, d_model)),
    } for _ in range(num_layers)]
    
    ff_weights = [{
        'linear1': nn.Parameter(torch.randn(d_model, dim_feedforward)),
        'linear2': nn.Parameter(torch.randn(dim_feedforward, d_model)),
    } for _ in range(num_layers)]
    
    norm_weights = [{
        'norm1': nn.Parameter(torch.ones(d_model)),
        'norm1_bias': nn.Parameter(torch.zeros(d_model)),
        'norm2': nn.Parameter(torch.ones(d_model)),
        'norm2_bias': nn.Parameter(torch.zeros(d_model)),
    } for _ in range(num_layers)]
    
    # Combine all parameters
    params = []
    for layer in range(num_layers):
        params.extend(list(attention_weights[layer].values()))
        params.extend(list(ff_weights[layer].values()))
        params.extend(list(norm_weights[layer].values()))
    
    # Convert to parameters
    params = nn.ParameterList(params)
    
    # Create optimizer
    optimizer = optim.Adam(params, lr=0.001)
    criterion = nn.MSELoss()
    
    # Create dataloader
    dataset = create_dummy_data()
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Training loop
    for epoch in range(num_epochs):
        total_loss = 0
        for batch_idx, (src, target) in enumerate(dataloader):
            # Forward pass through encoder
            x = src.transpose(0, 1)  # (seq_len, batch, d_model)
            
            for layer in range(num_layers):
                # Self-attention
                q = torch.matmul(x, attention_weights[layer]['q_proj'])
                k = torch.matmul(x, attention_weights[layer]['k_proj'])
                v = torch.matmul(x, attention_weights[layer]['v_proj'])
                
                attn_output, _ = multihead_attention(q, k, v, nhead)
                attn_output = torch.matmul(attn_output, attention_weights[layer]['out_proj'])
                
                x = x + F.dropout(attn_output, p=0.1)
                x = x * norm_weights[layer]['norm1'] + norm_weights[layer]['norm1_bias']
                
                # Feedforward
                ff_output = F.relu(torch.matmul(x, ff_weights[layer]['linear1']))
                ff_output = torch.matmul(ff_output, ff_weights[layer]['linear2'])
                
                x = x + F.dropout(ff_output, p=0.1)
                x = x * norm_weights[layer]['norm2'] + norm_weights[layer]['norm2_bias']
            
            # Mean pooling and loss calculation
            pooled = x.mean(dim=0)  # (batch, d_model)
            loss = criterion(pooled, target)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

In [29]:
train_transformer_encoder()

Epoch 1, Loss: 9532233443110182453248.0000
Epoch 2, Loss: 4349850475688186347520.0000
Epoch 3, Loss: 2580743977000428896256.0000
Epoch 4, Loss: 1695541153779045892096.0000
Epoch 5, Loss: 1431575931155340591104.0000
Epoch 6, Loss: 1105683419010879455232.0000
Epoch 7, Loss: 833641754458993328128.0000
Epoch 8, Loss: 742765210927705882624.0000
Epoch 9, Loss: 662325759532849561600.0000
Epoch 10, Loss: 636517876841863970816.0000


In [4]:
import math

In [31]:
def initialize_adaptive_attention(embed_dim, num_heads, max_len, adapt_span_params):
    params = {
        # Projection weights
        'q_proj_weight': nn.Parameter(torch.Tensor(embed_dim, embed_dim)),
        'k_proj_weight': nn.Parameter(torch.Tensor(embed_dim, embed_dim)),
        'v_proj_weight': nn.Parameter(torch.Tensor(embed_dim, embed_dim)),
        'out_proj_weight': nn.Parameter(torch.Tensor(embed_dim, embed_dim)),
        # Biases
        'q_proj_bias': nn.Parameter(torch.Tensor(embed_dim)),
        'k_proj_bias': nn.Parameter(torch.Tensor(embed_dim)),
        'v_proj_bias': nn.Parameter(torch.Tensor(embed_dim)),
        'out_proj_bias': nn.Parameter(torch.Tensor(embed_dim)),
        # Adaptive span
        'span_lim': nn.Parameter(torch.tensor(float(adapt_span_params['initial_span']))),
    }
    
    # Initialize weights
    for p in params.values():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
        else:
            nn.init.constant_(p, 0.)
    
    return params

In [32]:
def get_adaptive_span_mask(shape, span_lim, device):
    B, L = shape
    mask = torch.ones(L, L, dtype=torch.bool, device=device)
    mask = torch.tril(mask, diagonal=0)
    
    current_span = min(L, int(span_lim.item()))
    mask = mask * torch.ones_like(mask).triu(-current_span + 1)
    
    return mask.unsqueeze(0)

In [33]:
def adaptive_multihead_attention(
    query, key, value, 
    params, num_heads, 
    adapt_span_params, 
    key_padding_mask=None, 
    training=False
):
    B, T, _ = query.shape
    head_dim = params['q_proj_weight'].size(0) // num_heads
    
    # Project queries, keys, values
    q = F.linear(query, params['q_proj_weight'], params['q_proj_bias'])
    k = F.linear(key, params['k_proj_weight'], params['k_proj_bias'])
    v = F.linear(value, params['v_proj_weight'], params['v_proj_bias'])
    
    # Reshape for multi-head attention
    q = q.view(B, T, num_heads, head_dim).transpose(1, 2)
    k = k.view(B, T, num_heads, head_dim).transpose(1, 2)
    v = v.view(B, T, num_heads, head_dim).transpose(1, 2)
    
    # Compute attention scores
    attn_weights = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(head_dim)
    
    # Apply adaptive span mask
    span_mask = get_adaptive_span_mask((B, T), params['span_lim'], query.device)
    attn_weights = attn_weights.masked_fill(~span_mask, float('-inf'))
    
    # Apply key padding mask if provided
    if key_padding_mask is not None:
        attn_weights = attn_weights.masked_fill(
            key_padding_mask.unsqueeze(1).unsqueeze(2),
            float('-inf'))
    
    # Softmax
    attn_weights = F.softmax(attn_weights, dim=-1)
    attn_output = torch.matmul(attn_weights, v)
    
    # Combine heads
    attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, -1)
    attn_output = F.linear(attn_output, params['out_proj_weight'], params['out_proj_bias'])
    
    # Calculate span regularization loss if training
    span_loss = adapt_span_params['lambda'] * params['span_lim'] if training else 0
    
    return attn_output, span_loss

In [35]:
def initialize_encoder_layer(d_model, nhead, max_len, adapt_span_params, dim_feedforward=2048):
    params = {
        # Attention parameters
        **initialize_adaptive_attention(d_model, nhead, max_len, adapt_span_params),
        # Feedforward parameters
        'linear1_weight': nn.Parameter(torch.Tensor(dim_feedforward, d_model)),
        'linear1_bias': nn.Parameter(torch.Tensor(dim_feedforward)),
        'linear2_weight': nn.Parameter(torch.Tensor(d_model, dim_feedforward)),
        'linear2_bias': nn.Parameter(torch.Tensor(d_model)),
        # Layer norm parameters
        'norm1_weight': nn.Parameter(torch.ones(d_model)),
        'norm1_bias': nn.Parameter(torch.zeros(d_model)),
        'norm2_weight': nn.Parameter(torch.ones(d_model)),
        'norm2_bias': nn.Parameter(torch.zeros(d_model)),
    }

     # Initialize feedforward weights
    nn.init.kaiming_uniform_(params['linear1_weight'], a=math.sqrt(5))
    nn.init.kaiming_uniform_(params['linear2_weight'], a=math.sqrt(5))
    nn.init.constant_(params['linear1_bias'], 0.)
    nn.init.constant_(params['linear2_bias'], 0.)
    
    return params

In [36]:
def adaptive_encoder_layer(
    src, 
    params, 
    num_heads, 
    adapt_span_params, 
    dropout=0.1, 
    training=False,
    src_key_padding_mask=None
):
    """Functional adaptive transformer encoder layer"""
    # Self attention
    src2, span_loss = adaptive_multihead_attention(
        src, src, src, 
        params, num_heads, 
        adapt_span_params,
        key_padding_mask=src_key_padding_mask,
        training=training
    )
    src = src + F.dropout(src2, p=dropout)
    src = F.layer_norm(src, (src.size(-1),), params['norm1_weight'], params['norm1_bias'])
    
    # Feedforward
    src2 = F.linear(src, params['linear1_weight'], params['linear1_bias'])
    src2 = F.relu(src2)
    src2 = F.dropout(src2, p=dropout)
    src2 = F.linear(src2, params['linear2_weight'], params['linear2_bias'])
    src = src + F.dropout(src2, p=dropout)
    src = F.layer_norm(src, (src.size(-1),), params['norm2_weight'], params['norm2_bias'])
    
    return src, span_loss

In [37]:
def example_usage():
    # Configuration
    d_model = 512
    nhead = 8
    max_len = 1024
    dim_feedforward = 2048
    adapt_span_params = {
        'initial_span': 256,
        'lambda': 0.01,
        'nb_heads': nhead,
        'bs': 32
    }
    
    # Initialize parameters
    encoder_params = initialize_encoder_layer(
        d_model, nhead, max_len, adapt_span_params, dim_feedforward)
    
    # Create dummy input
    batch_size = 4
    seq_len = 100
    src = torch.randn(batch_size, seq_len, d_model)
    
    # Forward pass
    output, span_loss = adaptive_encoder_layer(
        src, encoder_params, nhead, adapt_span_params, training=True)
    
    print("Output shape:", output.shape)
    print("Span loss:", span_loss.item())

In [38]:
example_usage()

Output shape: torch.Size([4, 100, 512])
Span loss: 0.0


In [5]:
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [6]:
def generate_time_series(n=1000, seq_len=50):
    time = np.arange(0, n, 0.1)
    data = np.sin(time) + np.random.normal(scale=0.1, size=len(time))
    
    X, y = [], []
    for i in range(len(data) - seq_len - 1):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    
    X = np.array(X)[..., np.newaxis]
    y = np.array(y)
    return X, y

In [7]:
def positional_encoding(x, d_model, dropout=0.1, max_len=5000):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    pe = pe.unsqueeze(0).transpose(0, 1)
    x = x + pe[:x.size(0), :]
    return F.dropout(x, p=dropout)

In [8]:
def initialize_transformer_params(input_dim=1, d_model=64, nhead=4, num_layers=3, dim_feedforward=256):
    params = {
        'input_proj_weight': nn.Parameter(torch.Tensor(d_model, input_dim)),
        'input_proj_bias': nn.Parameter(torch.Tensor(d_model)),
        'decoder_weight': nn.Parameter(torch.Tensor(1, d_model)),
        'decoder_bias': nn.Parameter(torch.Tensor(1)),
    }
    
    # Initialize encoder layers
    for i in range(num_layers):
        params.update({
            f'encoder_{i}_self_attn_q_proj_weight': nn.Parameter(torch.Tensor(d_model, d_model)),
            f'encoder_{i}_self_attn_k_proj_weight': nn.Parameter(torch.Tensor(d_model, d_model)),
            f'encoder_{i}_self_attn_v_proj_weight': nn.Parameter(torch.Tensor(d_model, d_model)),
            f'encoder_{i}_self_attn_out_proj_weight': nn.Parameter(torch.Tensor(d_model, d_model)),
            f'encoder_{i}_linear1_weight': nn.Parameter(torch.Tensor(dim_feedforward, d_model)),
            f'encoder_{i}_linear2_weight': nn.Parameter(torch.Tensor(d_model, dim_feedforward)),
            f'encoder_{i}_norm1_weight': nn.Parameter(torch.ones(d_model)),
            f'encoder_{i}_norm1_bias': nn.Parameter(torch.zeros(d_model)),
            f'encoder_{i}_norm2_weight': nn.Parameter(torch.ones(d_model)),
            f'encoder_{i}_norm2_bias': nn.Parameter(torch.zeros(d_model)),
        })
    
    # Initialize weights
    for p in params.values():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
        else:
            nn.init.constant_(p, 0.)
    
    return params

In [9]:
def scaled_dot_product_attention(q, k, v, dropout_p=0.1):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    attn_weights = F.softmax(scores, dim=-1)
    attn_weights = F.dropout(attn_weights, p=dropout_p)
    return torch.matmul(attn_weights, v), attn_weights

In [16]:
def transformer_forward(x, params, nhead=4, num_layers=3, dropout=0.1):
    # Input projection
    x = F.linear(x, params['input_proj_weight'], params['input_proj_bias'])
    x = x.transpose(0, 1)  # (seq_len, batch, d_model)
    x = positional_encoding(x, x.size(-1), dropout)
    
    # Encoder layers
    for i in range(num_layers):
        # Self-attention
        q = F.linear(x, params[f'encoder_{i}_self_attn_q_proj_weight'])
        k = F.linear(x, params[f'encoder_{i}_self_attn_k_proj_weight'])
        v = F.linear(x, params[f'encoder_{i}_self_attn_v_proj_weight'])
        
        # Split into multiple heads
        batch_size, seq_len, d_model = q.size()
        assert d_model % nhead == 0, "d_model must be divisible by nhead"
        depth = d_model // nhead
        q = q.view(batch_size, seq_len, nhead, depth).transpose(1, 2)
        k = k.view(batch_size, seq_len, nhead, d_model // nhead).transpose(1, 2)
        v = v.view(batch_size, seq_len, nhead, d_model // nhead).transpose(1, 2)
        
        # Calculate attention
        attn_output, _ = scaled_dot_product_attention(q, k, v, dropout)
        
        # Combine heads
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, d_model)
        attn_output = F.linear(attn_output, params[f'encoder_{i}_self_attn_out_proj_weight'])
        
        x = x + F.dropout(attn_output, p=dropout)
        x = F.layer_norm(
            x, 
            normalized_shape=(x.size(-1),),
            weight=params[f'encoder_{i}_norm1_weight'],
            bias=params[f'encoder_{i}_norm1_bias']
        )
        
        # Feedforward
        ff_output = F.linear(x, params[f'encoder_{i}_linear1_weight'])
        ff_output = F.relu(ff_output)
        ff_output = F.linear(ff_output, params[f'encoder_{i}_linear2_weight'])
        
        x = x + F.dropout(ff_output, p=dropout)
        x = F.layer_norm(
            x,
            normalized_shape=(x.size(-1),),
            weight=params[f'encoder_{i}_norm2_weight'],
            bias=params[f'encoder_{i}_norm2_bias']
        )
    
    # Take last output and decode
    output = x[-1]  # (batch, d_model)
    return F.linear(output, params['decoder_weight'], params['decoder_bias'])

In [11]:
def initialize_lstm_params(input_dim=1, hidden_dim=64, num_layers=2):
    params = {}
    
    # LSTM parameters
    for layer in range(num_layers):
        input_size = input_dim if layer == 0 else hidden_dim
        for gate in ['i', 'f', 'g', 'o']:
            params[f'lstm_weight_ih_l{layer}_{gate}'] = nn.Parameter(torch.Tensor(hidden_dim, input_size))
            params[f'lstm_weight_hh_l{layer}_{gate}'] = nn.Parameter(torch.Tensor(hidden_dim, hidden_dim))
            params[f'lstm_bias_ih_l{layer}_{gate}'] = nn.Parameter(torch.Tensor(hidden_dim))
            params[f'lstm_bias_hh_l{layer}_{gate}'] = nn.Parameter(torch.Tensor(hidden_dim))
    
    # Output layer
    params['linear_weight'] = nn.Parameter(torch.Tensor(1, hidden_dim))
    params['linear_bias'] = nn.Parameter(torch.Tensor(1))
    
    # Initialize weights
    for p in params.values():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
        else:
            nn.init.constant_(p, 0.)
    
    return params

In [12]:
def lstm_forward(x, params, num_layers=2, dropout=0.1):
    batch_size, seq_len, input_dim = x.size()
    hidden_dim = params['linear_weight'].size(1)
    
    # Initialize hidden states
    h = [torch.zeros(batch_size, hidden_dim) for _ in range(num_layers)]
    c = [torch.zeros(batch_size, hidden_dim) for _ in range(num_layers)]
    
    # Process sequence
    for t in range(seq_len):
        xt = x[:, t, :]
        for layer in range(num_layers):
            # Get weights for this layer
            input_size = input_dim if layer == 0 else hidden_dim
            
            # Concatenated weights and biases
            W_ih = torch.cat([
                params[f'lstm_weight_ih_l{layer}_i'],
                params[f'lstm_weight_ih_l{layer}_f'],
                params[f'lstm_weight_ih_l{layer}_g'],
                params[f'lstm_weight_ih_l{layer}_o']
            ], dim=0)
            
            W_hh = torch.cat([
                params[f'lstm_weight_hh_l{layer}_i'],
                params[f'lstm_weight_hh_l{layer}_f'],
                params[f'lstm_weight_hh_l{layer}_g'],
                params[f'lstm_weight_hh_l{layer}_o']
            ], dim=0)
            
            b_ih = torch.cat([
                params[f'lstm_bias_ih_l{layer}_i'],
                params[f'lstm_bias_ih_l{layer}_f'],
                params[f'lstm_bias_ih_l{layer}_g'],
                params[f'lstm_bias_ih_l{layer}_o']
            ], dim=0)
            
            b_hh = torch.cat([
                params[f'lstm_bias_hh_l{layer}_i'],
                params[f'lstm_bias_hh_l{layer}_f'],
                params[f'lstm_bias_hh_l{layer}_g'],
                params[f'lstm_bias_hh_l{layer}_o']
            ], dim=0)
            
            # Compute all gates at once
            gates = F.linear(xt, W_ih, b_ih) + F.linear(h[layer], W_hh, b_hh)
            
            # Split into input, forget, cell, and output gates
            i, f, g, o = gates.chunk(4, 1)
            
            i = torch.sigmoid(i)
            f = torch.sigmoid(f)
            g = torch.tanh(g)
            o = torch.sigmoid(o)
            
            c[layer] = f * c[layer] + i * g
            h[layer] = o * torch.tanh(c[layer])
            
            if layer < num_layers - 1:
                h[layer] = F.dropout(h[layer], p=dropout)
            
            xt = h[layer]  # Input to next layer
    
    # Final output
    return F.linear(h[-1], params['linear_weight'], params['linear_bias'])

In [17]:
def train_and_compare():
    # Generate data
    X, y = generate_time_series()
    X_train, X_test = X[:800], X[800:]
    y_train, y_test = y[:800], y[800:]
    
    # Convert to PyTorch tensors
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
    X_test_t = torch.FloatTensor(X_test)
    y_test_t = torch.FloatTensor(y_test).unsqueeze(1)
    
    # Initialize models
    transformer_params = initialize_transformer_params()
    lstm_params = initialize_lstm_params()
    
    # Combine all parameters
    all_params = list(transformer_params.values()) + list(lstm_params.values())
    optimizer = optim.Adam(all_params, lr=0.001)
    criterion = nn.MSELoss()
    
    # Training loop
    print("Training Transformer and LSTM...")
    for epoch in range(50):
        optimizer.zero_grad()
        
        # Transformer forward
        transformer_output = transformer_forward(X_train_t, transformer_params)
        transformer_loss = criterion(transformer_output, y_train_t)
        
        # LSTM forward
        lstm_output = lstm_forward(X_train_t, lstm_params)
        lstm_loss = criterion(lstm_output, y_train_t)
        
        # Combined loss
        total_loss = transformer_loss + lstm_loss
        total_loss.backward()
        optimizer.step()
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Transformer Loss: {transformer_loss.item():.4f}, LSTM Loss: {lstm_loss.item():.4f}")
    
    # Evaluation
    with torch.no_grad():
        transformer_preds = transformer_forward(X_test_t, transformer_params).numpy().flatten()
        lstm_preds = lstm_forward(X_test_t, lstm_params).numpy().flatten()
        
        transformer_mse = mean_squared_error(y_test, transformer_preds)
        lstm_mse = mean_squared_error(y_test, lstm_preds)
    
    print(f"\nTransformer Test MSE: {transformer_mse:.6f}")
    print(f"LSTM Test MSE: {lstm_mse:.6f}")
    
    # Plot results
    plt.figure(figsize=(12, 6))
    plt.plot(y_test, label='True Values')
    plt.plot(transformer_preds, label='Transformer Predictions')
    plt.plot(lstm_preds, label='LSTM Predictions')
    plt.legend()
    plt.title("Time Series Forecasting Comparison (Functional)")
    plt.show()


In [18]:
train_and_compare()

Training Transformer and LSTM...
Epoch 0, Transformer Loss: 0.5048, LSTM Loss: 0.6400
Epoch 10, Transformer Loss: 0.5047, LSTM Loss: 0.1850
Epoch 20, Transformer Loss: 0.5047, LSTM Loss: 0.0715
Epoch 30, Transformer Loss: 0.5047, LSTM Loss: 0.0220
Epoch 40, Transformer Loss: 0.5047, LSTM Loss: 0.0176


RuntimeError: [enforce fail at alloc_cpu.cpp:115] data. DefaultCPUAllocator: not enough memory: you tried to allocate 66963360800 bytes.